In [93]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [94]:
## Load the trained model, scaler pickle file, one hot
model = load_model('model.h5')

In [95]:
with open('one_hot_encoder.pkl', 'rb') as file:
  one_hot_encoder = pickle.load(file)   ## As geography is multi-class and one-hot encoder was used for it

with open('label_encoder_gender.pkl', 'rb') as file:
  label_encoder_gender = pickle.load(file) ## As gender is binary-class and label encoder was used for it

with open('scaler.pkl', 'rb') as file:
  scaler = pickle.load(file)

In [96]:
## Example input data
input_data =  {
    "CreditScore": 619,
    "Geography": "France",
    "Gender": "Female",
    "Age": 42,
    "Tenure": 2,
    "Balance": 60000,
    "NumOfProducts": 1,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 101348.88,
}

In [97]:
input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,Female,42,2,60000,1,1,1,101348.88


### Now we will apply lable-encoder, one-hot-encoder and scaler for the data to convert into numerical format

In [98]:
geo_encoded = one_hot_encoder.transform([input_df['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=one_hot_encoder.get_feature_names_out(['Geography']))
geo_encoded_df

d:\DATA SCIENCE ML AI\DEEP_LEARNING\.venv\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [99]:
input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,0,42,2,60000,1,1,1,101348.88


In [ ]:
## concatination
input_df = pd.concat([input_df.drop('Geography', axis=1), geo_encoded_df], axis=1) ## column wise Geography drop, then column wise concatinate

In [101]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,60000,1,1,1,101348.88,1.0,0.0,0.0


In [ ]:
## Scale
input_df_scaled = scaler.transform(input_df)
input_df_scaled

array([[-0.33880827, -1.09499335,  0.29493847, -1.04241787, -0.25781119,
        -0.91668767,  0.64920267,  0.97481699,  0.01595384,  1.00150113,
        -0.57946723, -0.57638802]])

In [105]:
pd.DataFrame(input_df_scaled)

,0,1,2,3,4,5,6,7,8,9,10,11
0,-0.338808,-1.094993,0.294938,-1.042418,-0.257811,-0.916688,0.649203,0.974817,0.015954,1.001501,-0.579467,-0.576388


In [103]:
## Prediction
prediction = model.predict(input_df_scaled)
prediction

1/1 [==============================] - 0s 68ms/step


array([[0.33908716]], dtype=float32)

In [104]:
prediction_prob = prediction[0][0]
prediction_prob

0.33908716

### Now in the output layer we have "sigmoid", which gives 0.33908716 as output. For sigmoid threshold is 0.5, so 0 if < 0.5 else 1

In [108]:
if prediction_prob > 0.5: ## sigmoid threshold is 0.5.
  exited = 1
  print("Exited the bank")
else:
  exited = 0
  print("Did not exit the bank.")

Did not exit the bank.
